# Item 4 - The repetition direction, by hand

**Question.** Astra's edits act on one "repetition direction" in the residual stream (item 2). Can we find that
direction **without an LLM**, with the standard interpretability recipe, and does editing along it reproduce p50c?

1. **Practice generations.** The unedited model answers practice problems (GSM8K / MATH *train*, never the test
   sets) with Exp 8's thinking setting, capped at 8,192 tokens (loops start early: median at ~8 % of the answer).
2. **Contexts.** From each looping answer: a prefix that ends inside the loop (`rep`) and a matched prefix of the
   same answer that ends before the loop starts (`prod`). Finished answers give an unmatched control (`fin`).
3. **Activations** (one forward pass per context, transformers): the residual stream after layers 8-22 and the
   MLP neurons of layers 15 and 19, averaged over the last 32 tokens.
4. **Direction** = mean(rep) - mean(prod), per layer. Compare with Astra's p10c / p50c directions, check that it
   separates held-out contexts, check split-half stability.
5. **Edit** with p50c's exact form, `W = W0 - alpha * u (mask * u^T W0)` on the 64 neurons of layers 15 and 19 that
   write most along u, plus a norm-matched random-direction control. First a cheap teacher-forced test on held-out
   contexts (does the edit lower the log-prob of looping continuations only?), then a full generation on the same
   test problems as item 3.

Same setup as item 3 (A100; nothing to upload). It reuses item 3's test problems and, if present,
its `base | think35` and `p50c | think35` generations.

In [ ]:
!pip install -q -U vllm "transformers>=5.17" "math-verify[antlr4_13_2]" datasets
# fast kernels for the linear-attention layers (optional: without them transformers falls back to slower PyTorch code)
!pip install -q -U flash-linear-attention
!nvidia-smi --query-gpu=name,memory.total --format=csv
# If an import error mentions torch/transformers versions: Runtime -> Restart session, then continue from the next cell.

The helper module (the same as `analysis/colab/wsp_colab.py`), written to the VM.

In [ ]:
%%writefile /content/wsp_colab.py
"""Helpers for items 3 and 4 (Colab).  Put this file next to the notebooks in your Drive folder.

Item 3: decoding baselines - is Exp 8's gain something a sampling setting / non-thinking mode already gives?
Item 4: derive the "repetition direction" by hand (diff of means, no LLM), compare it with Astra's, and test edits.

Generation runs in a subprocess (`python wsp_colab.py gen ...`) so that vLLM releases the GPU after each checkpoint.
Every result is written to the Drive folder and every step is resumable.
"""
import argparse
import gc
import glob
import json
import os
import random
import re
import shlex
import shutil
import subprocess
import sys
import time

import numpy as np

BASE_REPO = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"          # the revision Exp 8 edited
LOGS_URL = "https://github.com/RikiyaT/lm_understand-logs.git"
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."
P = "model.language_model."                                      # tensor-name prefix in the checkpoint

# vLLM sampling settings.  think35 / nothink35 are exactly Exp 8's evaluate.py SAMPLING_THINK35 / SAMPLING_NOTHINK35.
PROTOCOLS = {
    # Exp 8's evaluation: thinking on, the model card's thinking benchmark setting, 32,768 tokens
    "think35": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5, max_tokens=32768),
    # model card, "thinking mode for text tasks"
    "think_t1": dict(thinking=True, temperature=1.0, top_p=0.95, top_k=20, presence_penalty=1.5, max_tokens=32768),
    # think35 + a multiplicative repetition penalty
    "think35_rep": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5,
                        repetition_penalty=1.1, max_tokens=32768),
    # think35 without the presence penalty (model card: "precise coding" setting)
    "think_pp0": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=0.0, max_tokens=32768),
    # the model card's NON-thinking benchmark setting; non-thinking is Qwen3.5-0.8B's default mode
    "nothink35": dict(thinking=False, temperature=0.7, top_p=0.8, top_k=20, presence_penalty=1.5, max_tokens=8192),
    # item 4: practice generations only need the start of each answer (loops start early)
    "think35_8k": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5, max_tokens=8192),
}


# ===================================================================== setup
q = shlex.quote


def sh(cmd):
    """Run a shell command and stream its output into the notebook cell (a plain subprocess.run would send it to the
    kernel's own terminal, so a long generation would look frozen)."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)


def get_logs(dest="/content/lm_understand-logs"):
    """Sparse clone of the logs repo: exp8_math only (code, edits, practice problems)."""
    if not os.path.exists(os.path.join(dest, "exp8_math/code/apply_edit.py")):
        sh(f"git clone --depth 1 --filter=blob:none --sparse {LOGS_URL} {q(dest)}")
        sh(f"cd {q(dest)} && git sparse-checkout set exp8_math")
    return os.path.join(dest, "exp8_math")


def get_base(dest="/content/models/Qwen3.5-0.8B"):
    from huggingface_hub import snapshot_download
    snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=dest)
    return dest


def base_st(base):
    return glob.glob(os.path.join(base, "model*.safetensors"))[0]


def _copy_non_weights(base, out):
    os.makedirs(out, exist_ok=True)
    for f in os.listdir(base):
        src = os.path.join(base, f)
        if os.path.isfile(src) and not f.endswith(".safetensors"):
            shutil.copy(src, os.path.join(out, f))


def official_edit_ckpt(run, exp8, base, out):
    """A full checkpoint with Astra's evaluated edit of `run` (p10c / p25c / p50c), rebuilt bit-exactly and
    hash-checked by the release's apply_edit.py."""
    st = base_st(base)
    target = os.path.join(out, os.path.basename(st))
    if not os.path.exists(os.path.join(out, "DONE")):
        _copy_non_weights(base, out)
        sh(f"{q(sys.executable)} {q(exp8 + '/code/apply_edit.py')} {q(st)} {q(exp8 + '/runs/' + run + '/edit')} {q(target)}")
        open(os.path.join(out, "DONE"), "w").write(run)
    return out


def make_ckpt(base, out, replace):
    """A full checkpoint = base with tensors replaced.  replace: {full tensor name: float array}; stored as bf16."""
    import torch
    from safetensors.torch import load_file, save_file
    if os.path.exists(os.path.join(out, "DONE")):
        return out
    _copy_non_weights(base, out)
    st = base_st(base)
    T = load_file(st)
    for k, v in replace.items():
        assert k in T and tuple(T[k].shape) == tuple(v.shape), k
        T[k] = torch.as_tensor(np.asarray(v, dtype=np.float32)).to(torch.bfloat16).contiguous()
    save_file(T, os.path.join(out, os.path.basename(st)), metadata={"format": "pt"})
    json.dump(sorted(replace), open(os.path.join(out, "DONE"), "w"))
    del T
    gc.collect()
    return out


# ===================================================================== grading (the release's own grader)
_GRADER = []


def grader(exp8):
    """exp8_math/code/mathqa.py: extract() = the last \\boxed{} after </think>, is_correct() = math-verify."""
    if not _GRADER:
        stub = os.path.join(os.path.dirname(os.path.abspath(__file__)), "_stubs")
        os.makedirs(stub, exist_ok=True)
        open(os.path.join(stub, "mcqa.py"), "w").write(
            "def loop_signals(text):\n    return {'repetitive_end': False}\n")   # only used by a function we skip
        for p in (stub, os.path.join(exp8, "code")):
            if p not in sys.path:
                sys.path.insert(0, p)
        import mathqa
        _GRADER.append(mathqa)
    return _GRADER[0]


# ===================================================================== problems
def _write_jsonl(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    os.replace(tmp, path)


def read_jsonl(path):
    return [json.loads(l) for l in open(path)]


def load_test(path, n_gsm=200, n_math=200, seed=0):
    """A fixed random subset of GSM8K test and MATH-500 (the benchmarks of the post), saved once to `path`."""
    if os.path.exists(path):
        return read_jsonl(path)
    from datasets import load_dataset
    rng = random.Random(seed)
    g = load_dataset("openai/gsm8k", "main", split="test")
    m = load_dataset("HuggingFaceH4/MATH-500", split="test")
    rows = []
    for i in sorted(rng.sample(range(len(g)), n_gsm)):
        rows.append(dict(id=f"gsm8k_test_{i}", source="gsm8k", problem=g[i]["question"],
                         answer=g[i]["answer"].split("####")[-1].strip().replace(",", "")))
    for i in sorted(rng.sample(range(len(m)), n_math)):
        rows.append(dict(id=m[i]["unique_id"], source="math500", problem=m[i]["problem"], answer=m[i]["answer"]))
    _write_jsonl(path, rows)
    return rows


def load_practice(path, exp8, n_gsm=150, n_math=150, seed=1):
    """Practice problems Astra could use (GSM8K train, MATH train) - never the test sets."""
    if os.path.exists(path):
        return read_jsonl(path)
    rng = random.Random(seed)
    rows = []
    for name, n in (("gsm8k_train", n_gsm), ("math_train", n_math)):
        allr = read_jsonl(os.path.join(exp8, "practice", f"{name}.jsonl"))
        rows += [dict(id=r["id"], source=r["source"], problem=r["problem"], answer=r["answer"]) for r in rng.sample(allr, n)]
    _write_jsonl(path, rows)
    return rows


def prompt_text(tok, problem, thinking):
    return tok.apply_chat_template([{"role": "user", "content": problem + "\n" + INSTRUCTION}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=thinking)


# ===================================================================== generation (vLLM, in a subprocess)
def generate(ckpt, problems_path, protocol, out_path, n=1, gpu_mem=0.85, chunk=50):
    """Resumable: answers are appended chunk by chunk to out_path; problems already there are skipped."""
    done = {r["id"] for r in read_jsonl(out_path)} if os.path.exists(out_path) else set()
    todo = [r for r in read_jsonl(problems_path) if r["id"] not in done]
    if not todo:
        print(f"[skip] {out_path}: complete")
        return out_path
    t = time.time()
    sh(f"{q(sys.executable)} {q(os.path.abspath(__file__))} gen --ckpt {q(ckpt)} --problems {q(problems_path)} "
       f"--protocol {protocol} --out {q(out_path)} --n {n} --gpu_mem {gpu_mem} --chunk {chunk}")
    print(f"[done] {out_path} in {(time.time() - t) / 60:.1f} min")
    return out_path


def _gen_main(a):
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    cfg = dict(PROTOCOLS[a.protocol])
    thinking = cfg.pop("thinking")
    tok = AutoTokenizer.from_pretrained(a.ckpt)
    done = {r["id"] for r in read_jsonl(a.out)} if os.path.exists(a.out) else set()
    rows = [r for r in read_jsonl(a.problems) if r["id"] not in done]
    llm = None
    common = dict(model=a.ckpt, dtype="bfloat16", gpu_memory_utilization=a.gpu_mem,
                  max_model_len=cfg["max_tokens"] + 2048, seed=0)
    for extra in ({"language_model_only": True}, {"limit_mm_per_prompt": {"image": 0, "video": 0}}, {}):
        try:
            llm = LLM(**common, **extra)
            print("vLLM started with", extra or "no extra options", flush=True)
            break
        except (TypeError, ValueError) as e:      # option unknown to this vLLM version
            print("vLLM option", extra, "refused:", str(e)[:200], flush=True)
    if llm is None:
        raise RuntimeError("vLLM could not load the model; install a newer vLLM (see the install cell)")
    sp = SamplingParams(n=a.n, seed=0, **cfg)
    for s in range(0, len(rows), a.chunk):
        part = rows[s:s + a.chunk]
        t0 = time.time()
        outs = llm.generate([prompt_text(tok, r["problem"], thinking) for r in part], sp, use_tqdm=False)
        with open(a.out, "a") as f:
            for r, o in zip(part, outs):
                for k, x in enumerate(o.outputs):
                    f.write(json.dumps(dict(id=r["id"], source=r["source"], answer=r["answer"], sample=k,
                                            text=x.text, finish=x.finish_reason, n_tokens=len(x.token_ids))) + "\n")
        ntok = sum(len(x.token_ids) for o in outs for x in o.outputs)
        ncut = sum(x.finish_reason == "length" for o in outs for x in o.outputs)
        print(f"  {min(s + a.chunk, len(rows))}/{len(rows)} problems | this chunk: {time.time() - t0:.0f}s, "
              f"{ntok / max(time.time() - t0, 1e-9):.0f} tok/s, {ncut}/{sum(len(o.outputs) for o in outs)} cut", flush=True)


# ===================================================================== scoring
NGRAM = 8


def loop_stats(text):
    """Word-level loop measures (same as item 1).
    dup_tail: share of the last 500 word-8-grams seen earlier; onset_char: where the text turns into a loop (the
    earliest word from which >= 90% of 8-grams are repeats, the loop lasting >= 200 words), -1 if never;
    onset_rel: the same as a share of the words; period: words between the last 8-gram and its previous occurrence."""
    spans = [m.start() for m in re.finditer(r"\S+", text)]
    w = text.split()
    n = len(w) - NGRAM + 1
    tail = text[-2000:]
    ws_tail = len(tail) > 0 and sum(c.isspace() for c in tail) / len(tail) >= 0.9
    if n <= 0:
        return dict(words=len(w), dup_tail=0.0, onset_char=-1, onset_rel=np.nan, period=np.nan, ws_tail=ws_tail)
    seen, last = set(), {}
    dup = np.zeros(n, dtype=np.int8)
    dists = []
    for i in range(n):
        g = tuple(w[i:i + NGRAM])
        if g in seen:
            dup[i] = 1
            dists.append((i, i - last[g]))
        seen.add(g)
        last[g] = i
    suf = np.cumsum(dup[::-1])[::-1] / np.arange(n, 0, -1)
    ok = np.where((suf >= 0.9) & (np.arange(n) < n - 200))[0]
    onset = int(ok[0]) if len(ok) else -1
    after = [d for i, d in dists if onset >= 0 and i >= onset]
    return dict(words=len(w), dup_tail=float(dup[-500:].mean()), onset_char=spans[onset] if onset >= 0 else -1,
                onset_rel=onset / n if onset >= 0 else np.nan,
                period=float(np.median(after)) if after else np.nan, ws_tail=ws_tail)


def score_file(path, exp8, cache=True):
    """One row per answer: correct, cut (finish_reason == length), loop measures, kind."""
    import pandas as pd
    cp = path.replace(".jsonl", ".scored.csv")
    if cache and os.path.exists(cp) and os.path.getmtime(cp) > os.path.getmtime(path):
        return pd.read_csv(cp)
    Q = grader(exp8)
    recs = []
    for r in read_jsonl(path):
        pred, nbox = Q.extract(r["text"])
        st = loop_stats(r["text"])
        cut = r["finish"] == "length"
        kind = "finished" if not cut else "cut: whitespace" if st["ws_tail"] else \
            "cut: loop" if st["dup_tail"] >= 0.8 else "cut: no loop"
        recs.append(dict(id=r["id"], source=r["source"], sample=r["sample"], correct=int(Q.is_correct(pred, r["answer"])),
                         pred=pred, cut=int(cut), n_tokens=r["n_tokens"], closed_think=int("</think>" in r["text"]),
                         kind=kind, **st))
    df = pd.DataFrame(recs)
    if cache:
        df.to_csv(cp, index=False)
    return df


def summarize(dfs, baseline):
    """dfs: {condition: scored DataFrame}.  Accuracy with SE, cut / loop shares, tokens, paired gain vs baseline."""
    import pandas as pd
    b = dfs[baseline].set_index(["id", "sample"])["correct"]
    rows = []
    for name, d in dfs.items():
        x = d.set_index(["id", "sample"])["correct"]
        j = pd.concat([x, b], axis=1, keys=["c", "b"]).dropna()
        diff = j.c - j.b
        row = dict(condition=name, n=len(d), acc=d.correct.mean(), se=d.correct.std() / np.sqrt(len(d)),
                   cut=d.cut.mean(), loop=(d.kind == "cut: loop").mean(), mean_tokens=d.n_tokens.mean(),
                   gain_vs_base=diff.mean(), gain_se=diff.std() / np.sqrt(len(diff)) if len(diff) > 1 else np.nan,
                   rescued=int(((j.c == 1) & (j.b == 0)).sum()), lost=int(((j.c == 0) & (j.b == 1)).sum()))
        for s in sorted(d.source.unique()):
            row[f"acc_{s}"] = d[d.source == s].correct.mean()
        row["acc_if_finished"] = d[d.cut == 0].correct.mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index("condition")


# ===================================================================== item 4: contexts and activations
def build_contexts(gen_path, scored, seed=0):
    """From base-model practice generations: contexts that end INSIDE a loop ("rep") and matched contexts from the
    same answers that end well before the loop starts ("prod"); plus contexts from answers that finished
    ("fin", an unmatched productive control).  Each keeps the next 64 tokens of the real text (for the log-prob test)."""
    rng = random.Random(seed)
    gens = {(r["id"], r["sample"]): r for r in read_jsonl(gen_path)}
    ctx = []
    for _, s in scored.iterrows():
        g = gens[(s["id"], s["sample"])]
        t = g["text"]
        if s["kind"] == "cut: loop" and s["onset_char"] >= 1500:
            o = int(s["onset_char"])
            per = s["period"] if s["period"] == s["period"] else 20
            into = min(len(t) - 400, o + max(300, int(3 * per * 6)))       # ~3 loop periods (~6 chars / word)
            prod = int(o * rng.uniform(0.3, 0.8))
            for kind, end in (("rep", into), ("prod", prod)):
                ctx.append(dict(kind=kind, gid=f"{s['id']}#{s['sample']}", id=s["id"], prefix=t[:end],
                                cont=t[end:end + 600]))
        elif s["kind"] == "finished" and s["closed_think"]:
            close = t.find("</think>")
            if close > 1500:
                end = int(close * rng.uniform(0.2, 0.7))
                ctx.append(dict(kind="fin", gid=f"{s['id']}#{s['sample']}", id=s["id"], prefix=t[:end],
                                cont=t[end:end + 600]))
    return ctx


def tokenize_contexts(ctx, tok, problems, thinking=True, max_ctx_tokens=8000, cont_tokens=64):
    out = []
    pmap = {p["id"]: p for p in problems}
    for c in ctx:
        head = prompt_text(tok, pmap[c["id"]]["problem"], thinking)
        ids = tok(head + c["prefix"], add_special_tokens=False).input_ids
        cont = tok(c["cont"], add_special_tokens=False).input_ids[:cont_tokens]
        if len(ids) <= max_ctx_tokens and len(cont) == cont_tokens:
            out.append(dict(c, ids=ids, cont_ids=cont))
    return out


def load_hf(base, device="cuda"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(base)
    m = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16 if device == "cuda" else torch.float32)
    return m.to(device).eval(), tok


def collect_acts(model, ctxs, layers=tuple(range(8, 23)), neuron_layers=(15, 19), K=32, log_every=50):
    """Per context: residual stream after each layer in `layers` (hidden_states[L+1]) and the MLP neuron
    activations (input of down_proj) of `neuron_layers`, both averaged over the context's last K tokens."""
    import torch
    dev = next(model.parameters()).device
    store = {}
    hooks = [model.get_submodule(f"model.layers.{L}.mlp.down_proj").register_forward_hook(
        lambda mod, inp, out, L=L: store.__setitem__(L, inp[0][0, -K:].float().mean(0).cpu().numpy()))
        for L in neuron_layers]
    H = {L: [] for L in layers}
    N = {L: [] for L in neuron_layers}
    t = time.time()
    try:
        with torch.no_grad():
            for i, c in enumerate(ctxs):
                o = model(torch.tensor([c["ids"]], device=dev), output_hidden_states=True, use_cache=False)
                for L in layers:
                    H[L].append(o.hidden_states[L + 1][0, -K:].float().mean(0).cpu().numpy())
                for L in neuron_layers:
                    N[L].append(store[L])
                del o
                if (i + 1) % log_every == 0:
                    print(f"  {i + 1}/{len(ctxs)} contexts, {time.time() - t:.0f}s", flush=True)
    finally:
        for h in hooks:
            h.remove()
    return {L: np.stack(v) for L, v in H.items()}, {L: np.stack(v) for L, v in N.items()}


def diff_direction(X, kinds, pos="rep", neg="prod"):
    kinds = np.asarray(kinds)
    d = X[kinds == pos].mean(0) - X[kinds == neg].mean(0)
    return d / np.linalg.norm(d)


def astra_directions(exp8, base, work):
    """Residual directions of Astra's p10c (one per layer, 8-22) and p50c (layers 15, 19) edits: top left singular
    vector of (edited - base) for each tensor (rank one, see item 2)."""
    import torch
    from safetensors.torch import load_file
    out = {}
    W0 = None
    for run in ("p10c", "p50c"):
        d = os.path.join(work, "astra_edits", run)
        if not os.path.exists(d):
            sh(f"{q(sys.executable)} {q(exp8 + '/code/apply_edit.py')} {q(base_st(base))} {q(exp8 + '/runs/' + run + '/edit')} {q(d)}")
        if W0 is None:
            W0 = load_file(base_st(base))
        for f in sorted(glob.glob(f"{d}/*.npy")):
            n = os.path.basename(f)[:-4]
            if run == "p10c" and "mlp.down_proj" not in n:
                continue                                      # p10c: one direction per layer, shared by its tensors
            D = np.load(f).astype(np.float64) - W0[n].float().numpy().astype(np.float64)
            u = np.linalg.svd(D, full_matrices=False)[0][:, 0]
            out[(run, int(n.split("layers.")[1].split(".")[0]))] = u
    return out


def top_k_edit(W0, u, act_mean, k=64, alpha=6.0):
    """Astra's p50c form: W = W0 - alpha * u (mask * (u^T W0)), on the k neurons that write the most along u
    in the measured contexts (score_j = |mean activation_j * u . W0[:, j]|).  W0: (1024, 3584) float64."""
    w = u @ W0
    score = np.abs(act_mean * w)
    cols = np.argsort(-score)[:k]
    mask = np.zeros_like(w)
    mask[cols] = 1
    return W0 - alpha * np.outer(u, mask * w), cols


def random_control(W0, u, act_mean, k, alpha, seed=0):
    """A random unit direction orthogonal to u, same selection rule, scaled so that |dW| matches the u-edit's."""
    r = np.random.default_rng(seed).standard_normal(len(u))
    r -= (r @ u) * u
    r /= np.linalg.norm(r)
    We, _ = top_k_edit(W0, u, act_mean, k, alpha)
    Wr, cols = top_k_edit(W0, r, act_mean, k, 1.0)
    s = np.linalg.norm(We - W0) / np.linalg.norm(Wr - W0)
    return W0 + s * (Wr - W0), cols, r


def logprob_test(model, ctxs, edits=None):
    """Mean log-prob per token of each context's real 64-token continuation, with optional in-memory edits
    {layer: down_proj W (1024, 3584)}.  Teacher-forced: one forward pass per context."""
    import torch
    dev = next(model.parameters()).device
    saved = {}
    try:
        for L, W in (edits or {}).items():
            p = model.get_submodule(f"model.layers.{L}.mlp.down_proj").weight
            saved[L] = p.data.clone()
            p.data = torch.as_tensor(W, dtype=p.dtype, device=dev)
        res = []
        with torch.no_grad():
            for c in ctxs:
                ids = torch.tensor([c["ids"] + c["cont_ids"]], device=dev)
                lp = torch.log_softmax(model(ids, use_cache=False).logits[0, len(c["ids"]) - 1:-1].float(), -1)
                res.append(lp.gather(1, ids[0, len(c["ids"]):, None]).mean().item())
        return np.array(res)
    finally:
        for L, W in saved.items():
            model.get_submodule(f"model.layers.{L}.mlp.down_proj").weight.data = W


# ===================================================================== CLI (the generation subprocess)
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["gen"])
    ap.add_argument("--ckpt")
    ap.add_argument("--problems")
    ap.add_argument("--protocol")
    ap.add_argument("--out")
    ap.add_argument("--n", type=int, default=1)
    ap.add_argument("--gpu_mem", type=float, default=0.85)
    ap.add_argument("--chunk", type=int, default=50)
    _gen_main(ap.parse_args())

In [ ]:
# Results go to Google Drive when it can be mounted (survives the runtime); otherwise to /content/wsp on the VM
# (e.g. in VS Code's Colab extension, where drive.mount may not work): download them before the runtime ends.
USE_DRIVE = True
import os, sys, importlib
WORK = '/content/wsp'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK = '/content/drive/MyDrive/wsp'
    except Exception as e:
        print('Google Drive not available here:', type(e).__name__, str(e)[:200])
print('results folder:', WORK)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, '/content')
import wsp_colab as W; importlib.reload(W)
os.makedirs(f'{WORK}/item4', exist_ok=True)
import numpy as np, pandas as pd, collections
pd.options.display.float_format = '{:.3f}'.format

## Configuration

In [ ]:
N_PRACTICE_GSM, N_PRACTICE_MATH = 150, 150   # practice problems for building the direction
N_SAMPLES = 2                                # answers per practice problem (more contexts for the same cost)
LAYERS = list(range(8, 23))                  # residual streams to measure (p10c's range)
EDIT_LAYERS = (15, 19)                       # p50c's layers (direction from hidden_states[L+1], neurons of layer L)
K_TOKENS, MAX_CTX = 32, 8000
TOPK, ALPHAS = 64, (3.0, 6.0)                # p50c used top-64 and alpha 6
HELDOUT = 0.2                                # share of practice problems held out for the checks
GEN_EDITS = ['hand_a6', 'random_a6']         # edits to evaluate by generation (add 'hand_a3' if time allows)

In [ ]:
exp8 = W.get_logs(); base = W.get_base()
practice_path = f'{WORK}/item4/practice_problems.jsonl'
practice = W.load_practice(practice_path, exp8, N_PRACTICE_GSM, N_PRACTICE_MATH)
problems_path = f'{WORK}/problems_test.jsonl'
test = W.load_test(problems_path)            # the same file as item 3 (created here if item 3 has not run)
print(len(practice), 'practice problems;', len(test), 'test problems')

## Step 1 - practice generations (unedited model, thinking, 8,192 tokens)

In [ ]:
gen_path = f'{WORK}/item4/practice_base.jsonl'
W.generate(base, practice_path, 'think35_8k', gen_path, n=N_SAMPLES)
sc = W.score_file(gen_path, exp8)
print(sc.kind.value_counts())
print('loop onset (share of the answer): median', sc.onset_rel.median().round(3))

## Step 2 - contexts

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(base)
ctx = W.build_contexts(gen_path, sc)
ctx = W.tokenize_contexts(ctx, tok, practice, max_ctx_tokens=MAX_CTX)
rng = np.random.default_rng(0)
ids = sorted({c['id'] for c in ctx}); held = set(rng.choice(ids, int(len(ids) * HELDOUT), replace=False))
for c in ctx: c['split'] = 'held' if c['id'] in held else 'train'
print(pd.crosstab(pd.Series([c['kind'] for c in ctx], name='kind'), pd.Series([c['split'] for c in ctx], name='split')))
print('context length (tokens): median', int(np.median([len(c['ids']) for c in ctx])))

## Step 3 - activations (cached to Drive)

In [ ]:
import torch
act_path = f'{WORK}/item4/acts.npz'
m, _ = W.load_hf(base)
if os.path.exists(act_path):
    z = np.load(act_path); H = {L: z[f'H{L}'] for L in LAYERS}; N = {L: z[f'N{L}'] for L in EDIT_LAYERS}
else:
    H, N = W.collect_acts(m, ctx, layers=LAYERS, neuron_layers=EDIT_LAYERS, K=K_TOKENS)
    np.savez(act_path, **{f'H{L}': v for L, v in H.items()}, **{f'N{L}': v for L, v in N.items()})
kind = np.array([c['kind'] for c in ctx]); split = np.array([c['split'] for c in ctx])
assert len(kind) == len(H[LAYERS[0]]), 'contexts changed since the cache was written: delete acts.npz'

## Step 4 - the direction, and how it compares with Astra's

In [ ]:
tr = split == 'train'
U = {L: W.diff_direction(H[L][tr], kind[tr], 'rep', 'prod') for L in LAYERS}
U_fin = {L: W.diff_direction(H[L][tr], kind[tr], 'rep', 'fin') for L in LAYERS}
A = W.astra_directions(exp8, base, '/content')       # {('p10c', L) | ('p50c', 15 / 19): unit vector}

# split-half stability: two disjoint halves of the training problems
tr_ids = sorted({c['id'] for c in ctx if c['split'] == 'train'}); half = set(rng.choice(tr_ids, len(tr_ids) // 2, replace=False))
hmask = np.array([c['id'] in half for c in ctx])
def auc(pos, neg):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg])
hd = split == 'held'
rows = []
for L in LAYERS:
    a = W.diff_direction(H[L][tr & hmask], kind[tr & hmask]); b = W.diff_direction(H[L][tr & ~hmask], kind[tr & ~hmask])
    row = dict(layer=L, split_half=abs(a @ b), rep_vs_fin_dir=abs(U[L] @ U_fin[L]),
               cos_p10c=abs(U[L] @ A[('p10c', L)]),
               auc_held=auc(H[L][hd & (kind == 'rep')] @ U[L], H[L][hd & (kind == 'prod')] @ U[L]))
    if ('p50c', L) in A: row['cos_p50c'] = abs(U[L] @ A[('p50c', L)])
    rows.append(row)
D = pd.DataFrame(rows).set_index('layer'); D.to_csv(f'{WORK}/item4/directions.csv')
print('random |cos| in R^1024: 99.9% below 0.10')
D

In [ ]:
# Same comparison with the 20 residual dims that dominate Astra's directions removed (item 2's control)
allA = np.stack(list(A.values())); top = np.argsort(-(allA ** 2).mean(0))[:20]; keep = np.setdiff1d(np.arange(1024), top)
nz = lambda v: v[keep] / np.linalg.norm(v[keep])
print('top dims', top[:10].tolist())
print(pd.DataFrame({L: dict(cos_p10c=abs(nz(U[L]) @ nz(A[('p10c', L)])),
                            **({'cos_p50c': abs(nz(U[L]) @ nz(A[('p50c', L)]))} if ('p50c', L) in A else {}))
                    for L in LAYERS}).T)
# does Astra's p50c direction also separate held-out rep / prod contexts?
for L in EDIT_LAYERS:
    print(f'layer {L}: held-out AUC, hand u = {D.auc_held[L]:.3f}, Astra p50c u = '
          f"{auc(H[L][hd & (kind == 'rep')] @ A[('p50c', L)], H[L][hd & (kind == 'prod')] @ A[('p50c', L)]):.3f}")

## Step 5 - edits, and a teacher-forced check on held-out contexts
Mean log-prob per token of each held-out context's real next 64 tokens. A good edit lowers it for `rep` (the loop continuing) and leaves `prod` / `fin` (reasoning continuing) alone.

In [ ]:
from safetensors.torch import load_file
name = lambda L: f'{W.P}layers.{L}.mlp.down_proj.weight'
W0 = {L: m.get_submodule(f'model.layers.{L}.mlp.down_proj').weight.detach().double().cpu().numpy() for L in EDIT_LAYERS}
act_rep = {L: N[L][tr & (kind == 'rep')].mean(0) for L in EDIT_LAYERS}
EDITS, COLS = {}, {}
for a in ALPHAS:
    EDITS[f'hand_a{a:g}'] = {}
    for L in EDIT_LAYERS:
        EDITS[f'hand_a{a:g}'][L], COLS[(a, L)] = W.top_k_edit(W0[L], U[L], act_rep[L], TOPK, a)
EDITS['random_a6'] = {L: W.random_control(W0[L], U[L], act_rep[L], TOPK, 6.0, seed=L)[0] for L in EDIT_LAYERS}
EDITS['astra_p50c'] = {L: np.load(f'/content/astra_edits/p50c/{name(L)}.npy').astype(np.float64) for L in EDIT_LAYERS}
p50_cols = {L: np.where(np.abs(EDITS['astra_p50c'][L] - W0[L]).sum(0) > 0)[0] for L in EDIT_LAYERS}
for L in EDIT_LAYERS:
    print(f'layer {L}: hand top-{TOPK} neurons overlap Astra\'s 64 in {len(set(COLS[(6.0, L)]) & set(p50_cols[L]))} (chance ~1.1)')

heldc = [c for c in ctx if c['split'] == 'held']
hk = np.array([c['kind'] for c in heldc])
lp0 = W.logprob_test(m, heldc)
res = {}
for e, ed in EDITS.items():
    d = W.logprob_test(m, heldc, {L: ed[L] for L in EDIT_LAYERS}) - lp0
    res[e] = {k: d[hk == k].mean() for k in ('rep', 'prod', 'fin')}
LP = pd.DataFrame(res).T; LP.columns = [f'dlogp_{k}' for k in LP.columns]; LP.to_csv(f'{WORK}/item4/logprob_test.csv')
LP   # nats / token; Astra reported -0.10 (rep) vs -0.01 (productive) for its edit

## Step 6 - generation on the test problems (item 3's setting)

In [ ]:
del m; import gc; gc.collect(); torch.cuda.empty_cache()     # free the GPU for vLLM
def out_path(name): return f'{WORK}/item4/{name}__think35.jsonl'
for e in GEN_EDITS:
    ck = W.make_ckpt(base, f'/content/ckpts/{e}', {name(L): EDITS[e][L] for L in EDIT_LAYERS})
    W.generate(ck, problems_path, 'think35', out_path(e))
# baseline and Astra's edit: reuse item 3's generations, else make them
ref = {'base': f'{WORK}/item3/base__think35.jsonl', 'p50c': f'{WORK}/item3/p50c__think35.jsonl'}
if not os.path.exists(ref['base']): W.generate(base, problems_path, 'think35', ref['base'])
if not os.path.exists(ref['p50c']):
    W.generate(W.official_edit_ckpt('p50c', exp8, base, '/content/ckpts/p50c'), problems_path, 'think35', ref['p50c'])

In [ ]:
dfs = {'base | think35': W.score_file(ref['base'], exp8), 'astra p50c | think35': W.score_file(ref['p50c'], exp8)}
dfs.update({f'{e} | think35': W.score_file(out_path(e), exp8) for e in GEN_EDITS})
S = W.summarize(dfs, 'base | think35'); S.to_csv(f'{WORK}/item4/summary.csv')
S[['n', 'acc', 'se', 'acc_gsm8k', 'acc_math500', 'cut', 'loop', 'acc_if_finished', 'gain_vs_base', 'gain_se', 'rescued', 'lost']]

## How to read it
- **Step 4**: `cos_p10c` / `cos_p50c` well above 0.10 (also with the top dims removed) and a held-out AUC near 1
  mean the hand-made direction is the one Astra found. `split_half` near 1 means it is stable.
- **Step 5**: the hand edit should lower `dlogp_rep` much more than `dlogp_prod`; the random control should not.
- **Step 6**: if `hand_a6` gets close to `astra p50c` and `random_a6` stays near the baseline, Astra's edit is
  reproducible with a standard diff-of-means recipe - the LLM automated a known method rather than found
  something new. If the hand edit fails while the direction matches, the difference is in Astra's neuron choice
  or its alpha, which is itself worth a closer look.